In [ ]:
import pubchempy as pcp
from pathlib import Path
from typing import Dict
import time

def get_smiles_from_cid_file(filepath: str, output_file: str = None, batch_size: int = 100) -> dict[int, str]:
    """
    Read CIDs from a file and fetch their SMILES from PubChem efficiently using batch requests.
    
    Args:
        filepath: Path to file containing PubChem CIDs (space or newline separated)
        output_file: Optional path to save CID-SMILES pairs (one per line: "SMILES\tCID")
        batch_size: Number of CIDs to fetch per batch (max 100 for PubChem API)
    
    Returns:
        Dictionary mapping CID (int) to SMILES (str)
    """
    # Read the file and extract all CIDs
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Split by whitespace and convert to integers
    cids = [int(cid) for cid in content.split()]
    
    print(f"Found {len(cids)} CIDs in file")
    print(f"Fetching in batches of {batch_size}...")
    
    start_time = time.time()
    cid_to_smiles = {}
    
    # Process in batches
    for i in range(0, len(cids), batch_size):
        batch = cids[i:i + batch_size]
        batch_num = i // batch_size + 1
        total_batches = (len(cids) + batch_size - 1) // batch_size
        
        try:
            # Fetch batch using comma-separated CIDs
            compounds = pcp.get_compounds(batch, 'cid')
            
            for compound in compounds:
                cid_to_smiles[compound.cid] = compound.isomeric_smiles
            
            print(f"Batch {batch_num}/{total_batches}: Retrieved {len(compounds)}/{len(batch)} compounds")
            
        except Exception as e:
            print(f"Batch {batch_num}/{total_batches}: Error - {e}")
            # Fallback to individual requests for this batch
            print(f"  Retrying batch individually...")
            for cid in batch:
                try:
                    compounds = pcp.get_compounds(cid, 'cid')
                    if compounds:
                        cid_to_smiles[cid] = compounds[0].isomeric_smiles
                except Exception as e2:
                    print(f"  CID {cid}: Error - {e2}")
        
        # Small delay to respect API rate limits
        time.sleep(0.2)
    
    elapsed = time.time() - start_time
    print(f"\nCompleted in {elapsed:.2f} seconds")
    print(f"Successfully retrieved {len(cid_to_smiles)}/{len(cids)} SMILES ({100*len(cid_to_smiles)/len(cids):.1f}%)")
    
    # Save to file if output_file is specified
    if output_file:
        with open(output_file, 'w') as f:
            for cid in sorted(cid_to_smiles.keys()):
                f.write(f"{cid_to_smiles[cid]}\t{cid}\n")
        print(f"Saved results to {output_file}")
    
    return cid_to_smiles

In [ ]:
# Example usage - fetch SMILES and save to file
cid_smiles = get_smiles_from_cid_file("inchi_1_06_knownissues.txt", 
                                       output_file="inchi_1_06_knownissues_smiles.txt")